# CallWhisper-8k: GramVaani 100 GPU Benchmark

Run this notebook top-to-bottom on a Colab GPU runtime. It evaluates the fixed 100-file GramVaani slice and its native-8-kHz/high-rate splits with OpenAI Whisper medium, Whisper large-v3, and ARTPARK Vaani Hindi.

It writes per-run JSON plus `model_comparison_v2.md` and `model_comparison_v2.json` to Google Drive. It does not train a model and it never uploads audio to GitHub.

In [ ]:
# Run this notebook with Runtime > Change runtime type > T4 GPU.
# Your Drive must already contain: MyDrive/call-whisper/GV_Dev_5h/

import os
import subprocess
import sys
from pathlib import Path

# Do not place a URL-shaped string in this cell: some Colab paste flows rewrite it as Markdown.
slash = chr(47)
dot = chr(46)
scheme = ''.join(chr(code) for code in [104, 116, 116, 112, 115])
REPO_URL = scheme + ':' + slash * 2 + 'github' + dot + 'com' + slash + 'anshulLuhsna' + slash + 'CallWhisper-8k.git'
REPO_DIR = Path('/content/CallWhisper-8k')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/call-whisper')
DRIVE_RESULTS_DIR = DRIVE_PROJECT_DIR / 'results' / 'benchmark_v2'
RUN_NAME = 'gramvaani_100_colab_benchmark_v2_seed0'

# Keep False for the first run. Set True only to resume after an interruption.
SKIP_COMPLETED_RUNS = False

OPENAI_MODELS = ['medium', 'large-v3']
HF_MODELS = ['ARTPARK-IISc/whisper-medium-vaani-hindi']
MANIFESTS = [
    'datasets/manifests/gramvaani_dev_100.csv',
    'datasets/manifests/gramvaani_dev_100_8khz.csv',
    'datasets/manifests/gramvaani_dev_100_highrate.csv',
]

subprocess.run(['nvidia-smi'], check=True)
print('Python:', sys.version)
print('Models:', OPENAI_MODELS + HF_MODELS)


In [ ]:
# Install the exact runtime dependencies, then clone a clean copy of the repo.
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)

# A previous run may have left the kernel inside REPO_DIR. Leave it before deleting it.
os.chdir('/content')
if REPO_DIR.exists():
    subprocess.run(['rm', '-rf', str(REPO_DIR)], check=True)
clone = subprocess.run(
    ['git', '-c', 'http.version=HTTP/1.1', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
    text=True,
    capture_output=True,
)
if clone.returncode:
    print('Git stdout:\n', clone.stdout)
    print('Git stderr:\n', clone.stderr)
    raise RuntimeError('Git clone failed. The stderr above identifies the Colab network or Git error.')
print(clone.stdout + clone.stderr)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[colab]'
], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
print('Repo:', REPO_DIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True).strip())


In [ ]:
# Mount Drive and expose the local audio to the committed manifests.
from google.colab import drive

drive.mount('/content/drive')
drive_data = DRIVE_PROJECT_DIR / 'GV_Dev_5h'
repo_data = REPO_DIR / 'datasets' / 'GV_Dev_5h'

if not drive_data.exists():
    raise FileNotFoundError(
        f'Missing {drive_data}. Upload the GV_Dev_5h folder to MyDrive/call-whisper first.'
    )
if repo_data.exists() or repo_data.is_symlink():
    repo_data.unlink()
repo_data.symlink_to(drive_data, target_is_directory=True)

audio_dir = repo_data / 'Audio'
required = [audio_dir, repo_data / 'text', repo_data / 'mp3.scp']
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Incomplete GV_Dev_5h folder. Missing: ' + ', '.join(missing))

print('Linked:', repo_data, '->', drive_data)
print('MP3 files visible:', len(list(audio_dir.glob('*.mp3'))))


In [ ]:
# Validate every frozen manifest before downloading model weights.
import csv

for manifest in MANIFESTS:
    path = REPO_DIR / manifest
    rows = list(csv.DictReader(path.open(encoding='utf-8')))
    missing_audio = [row['audio_path'] for row in rows if not (REPO_DIR / row['audio_path']).exists()]
    print(f'{manifest}: rows={len(rows)} missing_audio={len(missing_audio)}')
    if missing_audio:
        raise FileNotFoundError(f'Missing audio, first example: {missing_audio[0]}')


In [ ]:
# Run all benchmark jobs. Output is streamed live, including per-file tqdm progress.
import time

RESULTS_DIR = REPO_DIR / 'results' / 'benchmark_v2'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def safe_model_name(model_id):
    return model_id.lower().replace('/', '_').replace('-', '_')

def run_streaming(cmd):
    print('\n' + '=' * 100, flush=True)
    print('RUN:', ' '.join(map(str, cmd)), flush=True)
    started = time.time()
    process = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        env={**os.environ, 'PYTHONPATH': str(REPO_DIR / 'src')},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    elapsed = (time.time() - started) / 60
    if return_code:
        raise RuntimeError(f'Benchmark command failed after {elapsed:.1f} minutes with exit code {return_code}.')
    print(f'Finished in {elapsed:.1f} minutes.', flush=True)

jobs = []
for model in OPENAI_MODELS:
    for manifest in MANIFESTS:
        output = RESULTS_DIR / f'colab_whisper_{model}_{Path(manifest).stem}_seed0.json'
        jobs.append((output, [
            sys.executable, '-m', 'callwhisper.eval',
            '--manifest', manifest, '--model', model, '--language-mode', 'manifest',
            '--seed', '0', '--output-json', str(output),
        ]))

for model_id in HF_MODELS:
    for manifest in MANIFESTS:
        output = RESULTS_DIR / f'colab_hf_{safe_model_name(model_id)}_{Path(manifest).stem}_seed0.json'
        jobs.append((output, [
            sys.executable, '-m', 'callwhisper.eval.hf_runner',
            '--manifest', manifest, '--model-id', model_id, '--language-mode', 'manifest',
            '--seed', '0', '--output-json', str(output),
        ]))

print(f'Queued {len(jobs)} jobs: {len(OPENAI_MODELS)} OpenAI models + {len(HF_MODELS)} Hindi-tuned model(s), across {len(MANIFESTS)} slices.', flush=True)
for index, (output, cmd) in enumerate(jobs, start=1):
    print(f'\nJOB {index}/{len(jobs)} -> {output.name}', flush=True)
    if SKIP_COMPLETED_RUNS and output.exists():
        print('Already present; skipped.', flush=True)
        continue
    run_streaming(cmd)


In [ ]:
# Produce a compact, report-ready comparison and save every result to Drive.
import json
import shutil

import pandas as pd

rows = []
for path in sorted(RESULTS_DIR.glob('*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    summary = payload['summary']
    sample = payload['samples'][0]
    rows.append({
        'file': str(path.relative_to(REPO_DIR)),
        'model': sample['model'],
        'slice': sample['slice'],
        'condition': sample['condition'],
        'files': summary['num_files'],
        'wer': round(summary['wer'], 4),
        'cer': round(summary['cer'], 4),
    })

comparison = pd.DataFrame(rows).sort_values(['slice', 'model']).reset_index(drop=True)
if len(comparison) != len(jobs):
    raise RuntimeError(f'Expected {len(jobs)} result files, found {len(comparison)}. Do not interpret this as a complete benchmark.')

md_path = REPO_DIR / 'results' / 'model_comparison_v2.md'
json_path = REPO_DIR / 'results' / 'model_comparison_v2.json'
csv_path = REPO_DIR / 'results' / 'model_comparison_v2.csv'
md_path.write_text('# Model Comparison v2\n\n' + comparison.to_markdown(index=False) + '\n', encoding='utf-8')
json_path.write_text(comparison.to_json(orient='records', force_ascii=False, indent=2), encoding='utf-8')
comparison.to_csv(csv_path, index=False)

DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for path in RESULTS_DIR.glob('*.json'):
    shutil.copy2(path, DRIVE_RESULTS_DIR / path.name)
for path in [md_path, json_path, csv_path]:
    shutil.copy2(path, DRIVE_RESULTS_DIR / path.name)

display(comparison)
print('\nSaved all JSON, Markdown, and CSV files to:', DRIVE_RESULTS_DIR)


## Result wording

Use this notebook’s `model_comparison_v2.md` or CSV in your report. A valid claim is: *On the fixed GramVaani 100-file slice, model X got WER A and CER B; on the native 8 kHz subset, it got WER C and CER D.*

Do not treat these fixed slices as a global Hindi ASR leaderboard.